# EE-1.2: Sources, superposition and Thevenin equivalents

## Objectives

At the end of this notebook you should be able to:

- model ideal and real **voltage and current sources**, and explain why a source's terminal voltage drops under load
- solve networks where some node voltages are **fixed** (sources, the slack bus) by partitioning the conductance matrix
- use **superposition** in linear networks — and show why it fails for power
- reduce any linear network, seen from two terminals, to a **Thevenin equivalent**, and connect it to grid strength and maximum power transfer

Three ideas from this lesson come back later. The fixed-voltage node is the **slack bus** of every power flow (M1.3, Module 04). Superposition is why grid operators can describe flows with **distribution factors** (PTDFs, Module 04). And the Thevenin resistance is the circuit version of **grid strength**: how much the voltage moves when something connects.

In [1]:
import numpy as np
import pandas as pd

### Voltage sources and current sources

An **ideal voltage source** keeps its voltage fixed, whatever current flows — the current is set by the network. An **ideal current source** keeps its current fixed — the voltage adapts. Real sources sit in between: a battery, a generator or a whole upstream grid behaves like an ideal voltage source **E** behind an **internal resistance R_int**. Its terminal voltage drops as it delivers current:

**V_terminal = E − I · R_int**

A generator with E = 100 V and R_int = 0.5 Ω:

In [2]:
E, R_int = 100.0, 0.5
rows = []
for I in [0, 5, 10, 20]:
    V = E - I * R_int
    rows.append({"current_A": I, "terminal_V": V, "delivered_W": V * I, "lost_inside_W": I**2 * R_int})
pd.DataFrame(rows)

   current_A  terminal_V  delivered_W  lost_inside_W
0          0       100.0          0.0            0.0
1          5        97.5        487.5           12.5
2         10        95.0        950.0           50.0
3         20        90.0       1800.0          200.0

At 20 A the terminal voltage has fallen by 10 % and 200 W are lost inside the source. Two lessons: a source's "voltage" depends on how hard it is loaded, and power is always lost on the way — even inside the source. The upstream grid seen from your house behaves the same way, which is why lights can dim briefly when a large motor starts.

### Networks with fixed node voltages

In EE-1.1 only the reference node had a known voltage (0 V). With an ideal voltage source, a second node voltage is **known** too, and its current becomes an **unknown**. Split the nodes into known (K) and unknown (U) voltages. The rows of **G · V = I** for the unknown nodes read

G_UU · V_U + G_UK · V_K = I_U  →  **G_UU · V_U = I_U − G_UK · V_K**

— a smaller linear system, solved as before. Afterwards, the currents injected at the fixed nodes follow from **I = G · V**. That is exactly the slack bus from M1.3: its voltage is fixed, and its injection is whatever the network needs.

The network: a 100 V source at node 1, loads at nodes 2 and 3, and a 2 A current source at node 3 — think of rooftop solar feeding into the far end of a feeder.

![Three-node DC network](images/ee1_2_three_node_network.svg)

In [3]:
BRANCHES = [(1, 2, 2.0), (2, 3, 3.0), (2, 0, 10.0), (3, 0, 20.0)]   # (from, to, resistance in ohm)


def full_conductance(branches, n_nodes):
    """Conductance matrix (S) including the reference node as an ordinary node."""
    G = np.zeros((n_nodes, n_nodes))
    for a, b, r in branches:
        g = 1.0 / r
        G[a, a] += g; G[b, b] += g; G[a, b] -= g; G[b, a] -= g
    return G


def solve_fixed(branches, n_nodes, injections_A, fixed_V):
    """Node voltages (V) and net injections (A) with some node voltages fixed."""
    G = full_conductance(branches, n_nodes)
    K = sorted(fixed_V)
    U = [i for i in range(n_nodes) if i not in fixed_V]
    V = np.zeros(n_nodes)
    V[K] = [fixed_V[k] for k in K]
    I = np.asarray(injections_A, dtype=float)
    V[U] = np.linalg.solve(G[np.ix_(U, U)], I[U] - G[np.ix_(U, K)] @ V[K])
    return V, G @ V


pv = np.array([0, 0, 0, 2.0])                  # 2 A injected at node 3
V, inj = solve_fixed(BRANCHES, 4, pv, {0: 0.0, 1: 100.0})
pd.DataFrame({"voltage_V": V, "net_injection_A": inj}).round(3).rename_axis("node")

      voltage_V  net_injection_A
node                            
0         0.000          -11.797
1       100.000            9.797
2        80.405            0.000
3        75.135            2.000

Read the injection column: the free nodes inject exactly what was specified (0 A at node 2, 2 A at node 3) — KCL holds. The voltage source at node 1 delivers 9.80 A, and the reference node absorbs the 11.80 A that returns through the loads. Now the energy balance: everything the two sources deliver must be turned into heat in the resistors.

In [4]:
p_source = V[1] * inj[1]
p_pv = V[3] * pv[3]
heat = sum((V[a] - V[b]) ** 2 / r for a, b, r in BRANCHES)
assert np.isclose(p_source + p_pv, heat), "energy balance violated"
print(f"voltage source {p_source:.1f} W + current source {p_pv:.1f} W = heat {heat:.1f} W")

voltage source 979.7 W + current source 150.3 W = heat 1130.0 W


### Superposition

Because every equation is **linear**, the response to several sources is the **sum** of the responses to each source acting alone. To let one source act alone, switch the others off:

- a voltage source off = **0 V** (a short circuit)
- a current source off = **0 A** (an open circuit)

Scenario A: only the 100 V source. Scenario B: only the 2 A injection, with the voltage source set to 0 V.

In [5]:
V_A, _ = solve_fixed(BRANCHES, 4, np.zeros(4), {0: 0.0, 1: 100.0})
V_B, _ = solve_fixed(BRANCHES, 4, pv, {0: 0.0, 1: 0.0})
assert np.allclose(V_A + V_B, V), "superposition must hold for voltages"
pd.DataFrame({"only_source_V": V_A, "only_injection_V": V_B, "sum_V": V_A + V_B, "both_V": V}).round(3).rename_axis("node")

      only_source_V  only_injection_V    sum_V   both_V
node                                                   
0             0.000             0.000    0.000    0.000
1           100.000             0.000  100.000  100.000
2            77.703             2.703   80.405   80.405
3            67.568             7.568   75.135   75.135

The columns add up exactly. Superposition is what makes large networks tractable: compute the effect of **one** unit injection at each node once, and the effect of any combination is a weighted sum. Grid operators do exactly this with **power transfer distribution factors** (PTDFs): how much of a 1 MW transfer from A to B flows over each line. You will build them in Module 04 — and a first version in this lesson's exercise.

### Power does not superpose

Power is **quadratic** in voltage and current (P = I²R), so adding the losses of the two scenarios is wrong. Compare branch by branch:

In [6]:
rows = []
for a, b, r in BRANCHES:
    loss = lambda volts: (volts[a] - volts[b]) ** 2 / r
    rows.append({"branch": f"{a}-{b}", "only_source_W": loss(V_A), "only_injection_W": loss(V_B),
                 "sum_of_both_W": loss(V_A) + loss(V_B), "actual_W": loss(V)})
pd.DataFrame(rows).set_index("branch").round(2)

        only_source_W  only_injection_W  sum_of_both_W  actual_W
branch                                                          
1-2            248.58              3.65         252.24    191.97
2-3             34.24              7.89          42.13      9.26
2-0            603.77              0.73         604.50    646.50
3-0            228.27              2.86         231.13    282.26

Look at line 2–3. Adding the scenarios predicts 42 W of losses; the real value is 9 W, more than four times less. The injection at node 3 pushes current **against** the flow coming from the source, so the line carries less current, and losses fall with the square of the current. That is the real effect of distributed generation on a feeder — and adding up losses gets it backwards. (Over the whole network the cross terms happen to cancel in this example, so the totals agree; never rely on that.)

The same quadratic law in its simplest form: double the source voltage and the losses quadruple.

In [7]:
V_200, _ = solve_fixed(BRANCHES, 4, np.zeros(4), {0: 0.0, 1: 200.0})
losses = lambda volts: sum((volts[a] - volts[b]) ** 2 / r for a, b, r in BRANCHES)
print(f"losses at 100 V: {losses(V_A):.1f} W, at 200 V: {losses(V_200):.1f} W, ratio {losses(V_200) / losses(V_A):.2f}")

losses at 100 V: 1114.9 W, at 200 V: 4459.5 W, ratio 4.00


### Thevenin equivalents

**Thevenin's theorem:** any linear network, seen from two terminals, behaves exactly like one voltage source **V_th** in series with one resistance **R_th**. Find them with two experiments at the terminals:

1. **Open circuit** — connect nothing: the terminal voltage is **V_th**.
2. **Short circuit** — connect the terminals directly: the current that flows is **I_sc = V_th / R_th**.

Take the network "seen from node 3": everything except the 20 Ω load and the injection at node 3.

In [8]:
network_seen_from_3 = [(1, 2, 2.0), (2, 3, 3.0), (2, 0, 10.0)]

V_open, _ = solve_fixed(network_seen_from_3, 4, np.zeros(4), {0: 0.0, 1: 100.0})
V_th = V_open[3]
_, inj_short = solve_fixed(network_seen_from_3, 4, np.zeros(4), {0: 0.0, 1: 100.0, 3: 0.0})
I_sc = -inj_short[3]                      # current flowing from the network into the short
R_th = V_th / I_sc
print(f"V_th = {V_th:.3f} V, I_sc = {I_sc:.3f} A, R_th = {R_th:.3f} ohm")
print(f"by hand: R_th = 3 + (2 x 10) / (2 + 10) = {3 + 2 * 10 / 12:.3f} ohm")

V_th = 83.333 V, I_sc = 17.857 A, R_th = 4.667 ohm
by hand: R_th = 3 + (2 x 10) / (2 + 10) = 4.667 ohm


By hand: switch the source off (0 V, a short), and node 3 sees 3 Ω in series with 2 Ω and 10 Ω in parallel — 4.667 Ω, as computed. And V_th = 83.3 V is the voltage divider 100 V × 10 / (2 + 10).

The test of an equivalent is whether it **behaves** the same. Connect different loads to both the full network and the two-element equivalent:

In [9]:
rows = []
for R_load in [5.0, 20.0, 50.0]:
    V_full, _ = solve_fixed(network_seen_from_3 + [(3, 0, R_load)], 4, np.zeros(4), {0: 0.0, 1: 100.0})
    rows.append({"R_load_ohm": R_load, "current_full_network_A": V_full[3] / R_load,
                 "current_thevenin_A": V_th / (R_th + R_load)})
check = pd.DataFrame(rows)
assert np.allclose(check.current_full_network_A, check.current_thevenin_A)
check.round(5)

   R_load_ohm  current_full_network_A  current_thevenin_A
0         5.0                 8.62069             8.62069
1        20.0                 3.37838             3.37838
2        50.0                 1.52439             1.52439

Identical, for every load. Whatever is behind the terminals — two resistors here, a continental grid in reality — the terminals cannot tell the difference.

### Grid strength and maximum power transfer

In power systems the Thevenin equivalent seen from a connection point measures **grid strength**:

- a **small R_th** (a *strong* grid) keeps the voltage steady when loads or generators change: the voltage moves by ΔV = R_th · ΔI
- the **short-circuit current** I_sc = V_th / R_th, and the matching short-circuit power, are how network operators state that strength — a large wind farm or a data centre is only allowed where the short-circuit level is high enough

The equivalent also gives the most power the network can ever deliver to a load at this point, reached when R_load = R_th:

**P_max = V_th² / (4 R_th)**

— the same formula as the maximum transfer Vs² / (4R) in M1.3, now for a whole network.

In [10]:
R_loads = np.linspace(0.5, 30, 600)
P_load = (V_th / (R_th + R_loads)) ** 2 * R_loads
best = R_loads[P_load.argmax()]
assert abs(best - R_th) < 0.1 and np.isclose(P_load.max(), V_th**2 / (4 * R_th), rtol=1e-4)
print(f"maximum power {P_load.max():.1f} W at R_load = {best:.2f} ohm  (formula: {V_th**2 / (4 * R_th):.1f} W at {R_th:.2f} ohm)")
print(f"voltage change at node 3 per extra ampere: {R_th:.2f} V/A")

maximum power 372.0 W at R_load = 4.69 ohm  (formula: 372.0 W at 4.67 ohm)
voltage change at node 3 per extra ampere: 4.67 V/A


A numerical scan finds the same maximum as the formula. At that point, though, half the power is lost in R_th — maximum *power* is not maximum *efficiency*. Grids therefore operate far below this limit, and the voltage sensitivity R_th is the more useful everyday number.

## Check your understanding

**Sources and fixed voltages**

1. In the network above, raise the injection at node 3 from 2 A to 12 A. Predict the sign of the current the voltage source delivers, then check. What does it mean physically?
2. Why does the voltage source's current appear as an *output* of `solve_fixed`, and how is that the same as the slack bus in M1.3?

**Superposition and Thevenin**

1. From the superposition scenario B, how much does node 3's voltage rise per ampere injected at node 3? Compare it with R_th = 4.67 Ω. Why is it smaller? (Hint: R_th was computed without something that is connected at node 3 when the injection flows in.)
2. A wind farm developer can connect at node 2 or node 3. Without computing: which point is stronger, and why? Then compute R_th at node 2 to check.